In [8]:
%pip install regex
%pip install emoji

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import pandas as pd
import numpy as np

import re
import regex
import emoji
import unicodedata


## Load dataset

In [10]:
data_path = 'du_lieu_reviews_kem_sao.csv'
df = pd.read_csv(data_path)

In [11]:
df.head()

,restaurant_name,stars,review_text
0,Cô Ba Phở bò,1,Phở sao lại ăn kèm kimchi?! Biết là có nhiều k...
1,Cô Ba Phở bò,1,"Quán này gần chợ Hàn, có máy lạnh mát, phục vụ..."
2,Cô Ba Phở bò,1,"Tôi đã đi du lịch vòng quanh Việt Nam 10 ngày,..."
3,Cô Ba Phở bò,1,"Thành thật mà nói, điểm đánh giá cao là do các..."
4,Cô Ba Phở bò,1,Không hiểu sao chỗ này lại nổi tiếng thế nhỉ.....


In [12]:
print(df.shape)

print(df.columns)

df.info()

(6358, 3)
Index(['restaurant_name', 'stars', 'review_text'], dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 6358 entries, 0 to 6357
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   restaurant_name  6358 non-null   str  
 1   stars            6358 non-null   int64
 2   review_text      6358 non-null   str  
dtypes: int64(1), str(2)
memory usage: 149.1 KB


## Check null values & duplicate values 

In [13]:
df.isnull().sum()

restaurant_name    0
stars              0
review_text        0
dtype: int64

In [14]:
df.duplicated().sum()

np.int64(0)

## Add restaurant_id and review_id

In [15]:
# Tạo restaurant_id: mỗi tên nhà hàng unique → 1 ID số nguyên
restaurant_id_map = {
    name: idx + 1
    for idx, name in enumerate(df['restaurant_name'].unique())
}
df['restaurant_id'] = df['restaurant_name'].map(restaurant_id_map)

# Tạo review_id: đánh số thứ tự trong từng nhà hàng, dạng "<restaurant_id>_<n>"
df['review_id'] = (
    df['restaurant_id'].astype(str)
    + '_'
    + (df.groupby('restaurant_id').cumcount() + 1).astype(str)
)

# Sắp xếp lại thứ tự cột
df = df[['restaurant_id', 'review_id', 'restaurant_name', 'stars', 'review_text']]

print(f"Số nhà hàng: {df['restaurant_id'].nunique()}")
print(f"Số review: {len(df)}")
df.head()

Số nhà hàng: 141
Số review: 6358


,restaurant_id,review_id,restaurant_name,stars,review_text
0,1,1_1,Cô Ba Phở bò,1,Phở sao lại ăn kèm kimchi?! Biết là có nhiều k...
1,1,1_2,Cô Ba Phở bò,1,"Quán này gần chợ Hàn, có máy lạnh mát, phục vụ..."
2,1,1_3,Cô Ba Phở bò,1,"Tôi đã đi du lịch vòng quanh Việt Nam 10 ngày,..."
3,1,1_4,Cô Ba Phở bò,1,"Thành thật mà nói, điểm đánh giá cao là do các..."
4,1,1_5,Cô Ba Phở bò,1,Không hiểu sao chỗ này lại nổi tiếng thế nhỉ.....


## Text cleaning

In [16]:
def preprocess_text(text):

    # unicode normalize
    text = unicodedata.normalize('NFC', str(text))

    # lowercase
    text = text.lower()

    # remove urls
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # remove emails
    text = re.sub(r'\S+@\S+', ' ', text)

    # remove emoji
    text = emoji.replace_emoji(text, replace=' ')

    # normalize repeated chars
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # remove special characters
    text =  regex.sub(r'[^\p{L}\p{N}\s\.]', ' ', text)

    # remove excessive whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [17]:
df['review_text'] = df['review_text'].apply(preprocess_text)

## Save file

In [18]:
save_path = './datasets/processed/processed_reviews.csv'

df.to_csv(
    save_path,
    index=False,
    encoding='utf-8-sig'
)

print("Saved successfully!")

Saved successfully!
